# Cppcheck Maintainability — Code Smells Count Raw Output Extraction

This notebook analyzes **C repositories** with **Cppcheck** and captures the complete raw tool output for maintainability code-smells metric derivation and validation.

**Default benchmark repository:** [redis/redis](https://github.com/redis/redis)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.


## Section 1 — Install Dependencies

Install open-source packages required for repository acquisition, static analysis, and result processing.

On Linux/Colab, Cppcheck is installed via `apt-get`. On Windows/macOS, ensure `cppcheck` is on PATH or bootstrap it under `../../runtimes/cppcheck/`.


In [ ]:
!pip install -q pandas gitpython jupyter

import platform
if platform.system() == 'Linux':
    !apt-get update -qq
    !apt-get install -y -qq cppcheck

!cppcheck --version


## Section 2 — Configuration

Set execution mode, repository source, and output location.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/redis/redis.git'

LOCAL_REPO_PATH = '/content/redis'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

WORKSPACE_DIR = './workspace'

PROJECT_ROOT = Path('../../').resolve()

STREAM_RAW_OUTPUT = True

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark (predictable code-smell outcomes):
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/c_code_smells_benchmark'


## Section 3 — Imports and Utility Functions

Modular helpers for logging, repository setup, Cppcheck execution, and code-smell extraction.


In [ ]:
from pathlib import Path

from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

C_FILE_EXTENSIONS = {".c", ".h"}
EXCLUDED_DIR_NAMES = {
    ".git", "build", "dist", "out", "bin", "vendor", "third_party", "docs", "tests",
}
CPPCHECK_EXCLUDE_ARGS = [
    "-i.git", "-ibuild", "-idist", "-iout", "-ibin", "-ivendor", "-ithird_party", "-idocs", "-itests",
]
CODE_SMELL_RULE_IDS = {
    "duplicateExpression", "variableScope", "functionStatic", "staticFunction", "constVariable",
    "unreadVariable", "unusedFunction", "unusedStructMember", "shadowVariable", "passedByValue",
    "knownConditionTrueFalse",
}
RESULTS_COLUMNS = ["file", "line", "severity", "id", "message", "verbose", "cwe"]
SMELLS_COLUMNS = ["file", "line", "severity", "rule_id", "message", "cwe"]
PROGRESS_RE = re.compile(r"(\d+)/(\d+) files checked")


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")

    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}")
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg)
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg)
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str | Path,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_c_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for file_path in repo_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in C_FILE_EXTENSIONS:
            continue
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        files.append(file_path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, c_files: list[Path]) -> dict[str, int]:
    total_size = sum(path.stat().st_size for path in c_files)
    directories = {
        path.parent for path in c_files if not should_exclude_path(path.relative_to(repo_path))
    }
    return {
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
    }


def save_c_file_list(c_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    pd.DataFrame(
        [{"absolute_path": str(path), "relative_path": str(path.relative_to(repo_path))} for path in c_files]
    ).to_csv(output_csv, index=False)


def resolve_cppcheck_executable(project_root: Path | None = None) -> Path:
    env_path = os.environ.get("CPPCHECK")
    if env_path:
        candidate = Path(env_path)
        if candidate.exists():
            return candidate.resolve()

    which = shutil.which("cppcheck")
    if which:
        return Path(which).resolve()

    roots = []
    if project_root is not None:
        roots.append(project_root)
    roots.append(Path.cwd().resolve())
    for root in roots:
        for relative in (
            Path("runtimes/cppcheck/PFiles/Cppcheck/cppcheck.exe"),
            Path("runtimes/cppcheck/PFiles/Cppcheck/cppcheck"),
            Path("runtimes/cppcheck/cppcheck.exe"),
            Path("runtimes/cppcheck/cppcheck"),
            Path("../../runtimes/cppcheck/PFiles/Cppcheck/cppcheck.exe"),
            Path("../../runtimes/cppcheck/PFiles/Cppcheck/cppcheck"),
        ):
            candidate = (root / relative).resolve()
            if candidate.exists():
                return candidate

    raise FileNotFoundError(
        "Cppcheck executable not found. Install Cppcheck (apt-get install cppcheck) or set CPPCHECK."
    )


def build_cppcheck_command(cppcheck_exe: Path, repo_path: Path, xml_output: bool = False) -> list[str]:
    command = [str(cppcheck_exe), "--enable=all", "--inconclusive", "--force", *CPPCHECK_EXCLUDE_ARGS]
    if xml_output:
        command.extend(["--xml", "--xml-version=2"])
    command.append(str(repo_path))
    return command


def run_cppcheck_command(command: list[str], logger: NotebookLogger, stream_raw: bool = False) -> tuple[str, str, int]:
    if stream_raw:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        stdout_lines: list[str] = []
        stderr_lines: list[str] = []
        assert process.stdout is not None
        assert process.stderr is not None
        while True:
            stdout_line = process.stdout.readline()
            stderr_line = process.stderr.readline()
            if stdout_line:
                stdout_lines.append(stdout_line)
                print(stdout_line, end="")
            if stderr_line:
                stderr_lines.append(stderr_line)
                print(stderr_line, end="")
            if not stdout_line and not stderr_line and process.poll() is not None:
                break
        return "".join(stdout_lines), "".join(stderr_lines), process.returncode or 0

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )
    return completed.stdout, completed.stderr, completed.returncode


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def parse_cppcheck_xml(xml_text: str, logger: NotebookLogger) -> list[dict[str, Any]]:
    if not xml_text.strip():
        return []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError as exc:
        logger.error(f"Failed to parse Cppcheck XML: {exc}")
        return []

    rows: list[dict[str, Any]] = []
    for error in root.findall(".//error"):
        rule_id = error.get("id", "")
        if rule_id == "checkersReport":
            continue
        locations = error.findall("location")
        if not locations:
            rows.append({
                "file": error.get("file0", ""),
                "line": "",
                "severity": error.get("severity", ""),
                "id": rule_id,
                "message": error.get("msg", ""),
                "verbose": error.get("verbose", ""),
                "cwe": error.get("cwe", ""),
            })
            continue
        for location in locations:
            rows.append({
                "file": location.get("file", error.get("file0", "")),
                "line": location.get("line", ""),
                "severity": error.get("severity", ""),
                "id": rule_id,
                "message": error.get("msg", ""),
                "verbose": error.get("verbose", ""),
                "cwe": error.get("cwe", ""),
            })
    return rows


def findings_to_dataframe(findings: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(findings, columns=RESULTS_COLUMNS)


def is_code_smell(rule_id: str) -> bool:
    return rule_id in CODE_SMELL_RULE_IDS


def extract_code_smells_findings(findings: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen: set[tuple[str, str, str, str]] = set()
    for finding in findings:
        rule_id = str(finding.get("id", ""))
        if not is_code_smell(rule_id):
            continue
        key = (
            str(finding.get("file", "")),
            str(finding.get("line", "")),
            rule_id,
            str(finding.get("message", "")),
        )
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "file": finding.get("file", ""),
            "line": finding.get("line", ""),
            "severity": finding.get("severity", ""),
            "rule_id": rule_id,
            "message": finding.get("message", ""),
            "cwe": finding.get("cwe", ""),
        })
    return pd.DataFrame(rows, columns=SMELLS_COLUMNS)


def parse_progress_stats(stdout: str, total_files: int) -> tuple[int, int]:
    checked = 0
    total = total_files
    for match in PROGRESS_RE.finditer(stdout):
        checked = int(match.group(1))
        total = int(match.group(2))
    if checked == 0 and total_files > 0:
        checked = total_files
        total = total_files
    return checked, max(total - checked, 0)


def count_failed_files(findings: list[dict[str, Any]]) -> int:
    failure_ids = {"syntaxError", "internalError", "unknownMacro", "preprocessorErrorDirective"}
    failed_files = {
        str(item.get("file", ""))
        for item in findings
        if str(item.get("id", "")) in failure_ids and str(item.get("file", ""))
    }
    return len(failed_files)


def compute_code_smells_summary(findings_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([{"metric_name": "Code_Smells_Count", "metric_value": len(findings_df)}])


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW CPPCHECK OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


## Section 4 — Repository Setup

Resolve the repository path based on configuration and initialize output directories.


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

logger.info(f'Repository ready at: {REPO_PATH}')


## Section 5 — Discover C Files

Recursively discover `.c` and `.h` files while excluding build, vendor, and test directories.


In [ ]:
C_FILES = discover_c_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, C_FILES)

C_FILES_CSV = OUTPUT_PATH / 'c_files.csv'
save_c_file_list(C_FILES, REPO_PATH, C_FILES_CSV)

print(f'Total C Files Found: {len(C_FILES)}')
print(f'Repository Size (C files only): {REPO_STATS["repository_size_bytes"]:,} bytes')
print(f'Total Directories (excluding filtered paths): {REPO_STATS["directory_count"]:,}')
print(f'Saved file list to: {C_FILES_CSV}')


## Section 6 — Execute Cppcheck

Run Cppcheck against the repository. Execution continues even if individual files fail.

Example equivalent command:

```bash
cppcheck --enable=all --inconclusive --force --xml --xml-version=2 <repo_path>
```


In [ ]:
try:
    CPPCHECK_EXE = resolve_cppcheck_executable(PROJECT_ROOT)
    logger.info(f'Using Cppcheck executable: {CPPCHECK_EXE}')
except Exception as exc:
    logger.error(str(exc))
    raise

if not C_FILES:
    logger.error('No C files discovered; skipping Cppcheck execution.')
    CPPCHECK_RAW_TEXT = ''
    CPPCHECK_XML_TEXT = ''
    FILES_SUCCESS = 0
    FILES_FAILED = 0
    CPPCHECK_FINDINGS: list[dict] = []
else:
    text_cmd = build_cppcheck_command(CPPCHECK_EXE, REPO_PATH, xml_output=False)
    text_stdout, text_stderr, text_code = run_cppcheck_command(text_cmd, logger, stream_raw=STREAM_RAW_OUTPUT)
    CPPCHECK_RAW_TEXT = combine_raw_streams(text_stdout, text_stderr)
    if text_code not in (0, 1):
        logger.error(f'Cppcheck text run exited with code {text_code}')

    xml_cmd = build_cppcheck_command(CPPCHECK_EXE, REPO_PATH, xml_output=True)
    xml_stdout, xml_stderr, xml_code = run_cppcheck_command(xml_cmd, logger, stream_raw=False)
    CPPCHECK_XML_TEXT = xml_stderr if xml_stderr.strip().startswith('<?xml') else xml_stderr
    if not CPPCHECK_XML_TEXT.strip().startswith('<?xml') and xml_stdout.strip().startswith('<?xml'):
        CPPCHECK_XML_TEXT = xml_stdout
    if xml_code not in (0, 1):
        logger.error(f'Cppcheck XML run exited with code {xml_code}')

    CPPCHECK_FINDINGS = parse_cppcheck_xml(CPPCHECK_XML_TEXT, logger)
    FILES_SUCCESS, FILES_FAILED = parse_progress_stats(text_stdout, len(C_FILES))
    FILES_FAILED = max(FILES_FAILED, count_failed_files(CPPCHECK_FINDINGS))
    if FILES_SUCCESS == 0 and len(C_FILES) > 0:
        FILES_SUCCESS = len(C_FILES) - FILES_FAILED

logger.info(f'Cppcheck execution complete. Files success={FILES_SUCCESS}, failed={FILES_FAILED}')


## Section 7 — Raw Output Extraction

Persist complete raw Cppcheck text output, XML output, and a CSV representation of all findings.


In [ ]:
RAW_OUTPUT_PATH = OUTPUT_PATH / 'cppcheck_raw_output.txt'
XML_OUTPUT_PATH = OUTPUT_PATH / 'cppcheck_output.xml'
RESULTS_CSV_PATH = OUTPUT_PATH / 'cppcheck_results.csv'

RAW_OUTPUT_PATH.write_text(CPPCHECK_RAW_TEXT, encoding='utf-8')
XML_OUTPUT_PATH.write_text(CPPCHECK_XML_TEXT, encoding='utf-8')

CPPCHECK_RESULTS_DF = findings_to_dataframe(CPPCHECK_FINDINGS)
CPPCHECK_RESULTS_DF.to_csv(RESULTS_CSV_PATH, index=False)

logger.info(f'Saved raw output: {RAW_OUTPUT_PATH}')
logger.info(f'Saved XML output: {XML_OUTPUT_PATH}')
logger.info(f'Saved CSV results: {RESULTS_CSV_PATH}')
logger.info(f'Total Cppcheck findings: {len(CPPCHECK_RESULTS_DF)}')

preview_raw_output(CPPCHECK_RAW_TEXT, RAW_OUTPUT_PREVIEW_LINES, RAW_OUTPUT_PATH)


## Section 8 — Code Smell Extraction

Extract maintainability-related findings including duplicateExpression, variableScope, functionStatic, constVariable, unreadVariable, unusedFunction, unusedStructMember, shadowVariable, passedByValue, and knownConditionTrueFalse.


In [ ]:
CODE_SMELLS_DF = extract_code_smells_findings(CPPCHECK_FINDINGS)
CODE_SMELLS_CSV = OUTPUT_PATH / 'code_smells_findings.csv'
CODE_SMELLS_DF.to_csv(CODE_SMELLS_CSV, index=False)

logger.info(f'Saved code smells findings: {CODE_SMELLS_CSV}')
logger.info(f'Code smells count: {len(CODE_SMELLS_DF)}')

if not CODE_SMELLS_DF.empty:
    display(CODE_SMELLS_DF.head(15))
else:
    print('No code smell findings detected.')


## Section 9 — Metric Computation

Compute repository-level code smells count:

**Code_Smells_Count** = count(all maintainability-related findings)


In [ ]:
SUMMARY_DF = compute_code_smells_summary(CODE_SMELLS_DF)
SUMMARY_CSV = OUTPUT_PATH / 'code_smells_summary.csv'
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f'Saved code smells summary: {SUMMARY_CSV}')
display(SUMMARY_DF)


## Section 10 — Summary Dashboard

Overview of analysis coverage, Cppcheck findings, and code-smell metrics.


In [ ]:
code_smells_count = int(SUMMARY_DF.loc[SUMMARY_DF['metric_name'] == 'Code_Smells_Count', 'metric_value'].iloc[0])

summary_df = pd.DataFrame(
    [
        {'Metric': 'Total C Files', 'Value': len(C_FILES)},
        {'Metric': 'Files Successfully Analyzed', 'Value': FILES_SUCCESS},
        {'Metric': 'Files Failed', 'Value': FILES_FAILED},
        {'Metric': 'Total Cppcheck Findings', 'Value': len(CPPCHECK_RESULTS_DF)},
        {'Metric': 'Total Code Smells', 'Value': code_smells_count},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    XML_OUTPUT_PATH,
    RESULTS_CSV_PATH,
    C_FILES_CSV,
    CODE_SMELLS_CSV,
    SUMMARY_CSV,
    ERROR_LOG_PATH,
]

print('\nDeliverables:')
for deliverable in deliverables:
    status = 'OK' if deliverable.exists() else 'MISSING'
    print(f'  [{status}] {deliverable}')


## Section 11 — Error Handling

Failures encountered during cloning, validation, or Cppcheck execution are appended to `outputs/error_log.txt`.


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 12 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── cppcheck_raw_output.txt
├── cppcheck_output.xml
├── cppcheck_results.csv
├── c_files.csv
├── code_smells_findings.csv
├── code_smells_summary.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.
